# Transactions

A transaction commits all changes together or rolls them all back.

In [ ]:
# A transaction makes both balance changes succeed or fail together.
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER)")
db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 50)])
db.commit()

try:
    # Raising inside the context rolls back both updates automatically.
    with db:
        db.execute("UPDATE accounts SET balance = balance - 30 WHERE id = 1")
        db.execute("UPDATE accounts SET balance = balance + 30 WHERE id = 2")
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

print(db.execute("SELECT * FROM accounts").fetchall())

The balances remain unchanged because the transaction rolled back.

## Polished version

A request-scoped dependency owns the transaction. Success commits before the response; an exception rolls back before its error response.

In [ ]:
# This fake session exposes the same lifecycle used by AsyncSession.
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Annotated

import httpx
from fastapi import Depends, FastAPI, HTTPException

events: list[str] = []

class Session:
    async def __aenter__(self) -> "Session":
        events.append("open")
        return self

    async def __aexit__(self, *args: object) -> None:
        events.append("close")

    @asynccontextmanager
    async def begin(self) -> AsyncIterator[None]:
        events.append("begin")
        try:
            yield
        except Exception:
            events.append("rollback")
            raise
        else:
            events.append("commit")

    async def add_task(self, task_id: int) -> None:
        events.append(f"insert:{task_id}")


async def get_session() -> AsyncIterator[Session]:
    session = Session()
    # In production, session_factory returns a SQLAlchemy AsyncSession.
    async with session, session.begin():
        yield session


SessionDependency = Annotated[Session, Depends(get_session, scope="function")]
app = FastAPI()

@app.post("/tasks/{task_id}", status_code=201)
async def create_task(task_id: int, session: SessionDependency) -> dict[str, int]:
    if task_id <= 0:
        raise HTTPException(status_code=422, detail="Invalid task ID")
    await session.add_task(task_id)
    return {"id": task_id}

transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    created = await client.post("/tasks/1")
    rejected = await client.post("/tasks/0")

print(created.status_code, rejected.status_code)
print(events)

## Applied in this repository

The REST [session dependency](../00P1-project-rest-api/app/interface/dependencies.py) uses this boundary with SQLAlchemy. `scope="function"` completes commit or rollback before FastAPI sends the response.